# 实验目的
通过模型退化，验证PE模型正确性，分段线性位移变换正确性。
1. 将海浪高度，风速等条件设置为0，验证模型能否完美还原双射线模型解
2. 在距离发射源100m处添加单个斜坡，验证PE与FDTD模型的均方根误差

# 实验步骤
## 验证方案
抛物方程（Parabolic Equation, PE）方法自20世纪40年代由Leontovich和Fock提出，并在70年代由Tappert引入分步傅里叶变换（SSFT）求解后，已经成为解决大尺度、非均匀介质（如大气波导）中电磁波传播的“黄金标准” (Gold Standard)。因此不需要验证PE算法。
但是引入分段线性位移变换修正下边界，这是对原有模型的改进，需要验证是否引入了非物理的数值误差，遮挡效应计算是否准确。但是不需要进行物理实测验证，而是使用数值对标验证。
### 退化验证（实验一）
将海浪高度设为0（退化为平坦海面）。验证分段线性变换在平坦情况下是否能完美还原为标准PE的解（或双射线模型解）。但是需要解释为什么 PE 算出结果和双射线算出结果在图示上的传播效果有区别，并且得到两张图例
1. 2D 空间场强热力图对比
2. 1D 场强切片对比曲线
### 实验二：网格收敛性分析（PE 内部自证）场景： 

选取某一个中等海况（例如风速 $5\text{m/s}$ 生成的固定一维粗糙海面剖面）。

对比： 仅运行 PE 求解器。分别使用 $\Delta z = [0.8, 0.4, 0.2, 0.1, 0.05]\text{m}$ 跑同一个粗糙海面。以 $\Delta z = 0.05\text{m}$ 的结果作为“拟似真实解”，计算其他较粗网格相对它的 RMSE。

目的： 证明引入 PLST 处理粗糙海面时，算法是收敛的。

关键结论： 通过绘制一张 RMSE 随 $\Delta z$ 减小而收敛的曲线图，

得出结论：“当 $\Delta z \le 0.2\text{m}$ 时，相对误差变化率已小于 1%，兼顾了极高的计算效率与精度。因此，本文后续的所有复杂实验均采用 $\Delta z = 0.2\text{m}$。”(注：这一步极快，因为完全不需要跑 FDTD，纯粹是 PE 自己跟自己比。)

### 指标

a. 均方根误差 (RMSE) -- 核心指标
$$RMSE = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (L_{PE}(i) - L_{FDTD}(i))^2}$$
< 1 dB: 极好（Excellent），几乎完美复现全波解。

1 ~ 3 dB: 良好（Good），这是大多数改进型 PE 算法能达到的区间，完全可以接受。

\> 5 dB: 需要解释原因（例如只在深阴影区误差大，但在覆盖区很准）。



# 方法延申
验证了方法正确性后，引入编队场景。用PE算出编队中N艘船的功率分布情况，生成一个图（Dynamic Graph）。聚焦于恶劣海况下编队构型的稳健性分析

# 代码实现
1. 生成JONSWAP海面
2. 得到观测海面高度（x-z平面）
3. 分别进行PE FDTD计算
4. 绘制对比图

In [71]:
# 导入包
import meep as mp
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
import scipy.fft as fft
import pandas as pd
from abc import ABC, abstractmethod
import os
from scipy.ndimage import uniform_filter1d


# ==========================================
# 场景生成器 (Scene Generator)
# 定义全局物理参数、生成海面、统一下发坐标
# ==========================================
class SceneGenerator:
    def __init__(self, 
                 freq_ghz=0.3,      # 频率(GHz)
                 lx=320.0,          # 仿真域长度 (m) 
                 lz=50.0,           # 仿真域高度 (m)
                 dpml=5.0,          # 吸收层厚度 (m)
                 wind_speed=0.0,   # 风速 (m/s)
                 fetch_km=0.0,     # 风区 (km)
                 tx_height=5.0,     # 发射机高度 (相对海平面, m)
                 rx_height=5.0,
                 terrain_type='gaussian_hill'):    # 接收机观测高度 (相对海平面, m)
        self.freq_ghz = freq_ghz
        self.lx = lx        
        self.lz = lz
        self.dpml = dpml
        self.wind_speed = wind_speed
        self.fetch_km = fetch_km
        self.tx_height = tx_height
        self.rx_height = rx_height
        self.terrain_type = terrain_type
        self.base_water_level = 0# 绝对坐标系下的平均海平面位置
        
        # FDTD 网格分辨率计算
        self.resolution = 10        # FDTD 网格分辨率
        self.dx_fdtd = 1.0 / self.resolution
        
        self.x_full = np.arange(0, self.lx + 2 * self.dpml, self.dx_fdtd)
        self._generate_terrain()

    def _generate_terrain(self):
        """符合 OCP 原则的地形生成工厂"""
        if self.terrain_type == 'flat':
            # 退化验证：平坦海面
            self.h_full = np.zeros_like(self.x_full)
        elif self.terrain_type == 'gaussian_hill':
            # 规范地形验证：高斯山丘 
            hill_center = self.lx / 2.0
            hill_height = 1.0
            hill_width = 20.0
            self.h_full = hill_height * np.exp(-0.5 * ((self.x_full - hill_center) / hill_width)**2)
        else:
            raise ValueError("Unsupported terrain type")


In [72]:
# ==========================================
# 双射线求解接口 (TwoRaySolver)
# ==========================================
class TwoRaySolver:
    """双射线解析模型求解器 (针对 2D 柱面波传播)"""
    def __init__(self, scene: SceneGenerator):
        self.scene = scene
        self.k0 = 2 * np.pi * (scene.freq_ghz * 1e9) / 299792458.0

    def run(self, x_coords, z_coords):
        # 构建计算网格
        X, Z = np.meshgrid(x_coords, z_coords, indexing='ij')
        X_safe = np.maximum(X, 1e-3) # 防止除零
        
        # 直射径与反射径
        R1 = np.sqrt(X_safe**2 + (Z - self.scene.tx_height)**2)
        R2 = np.sqrt(X_safe**2 + (Z + self.scene.tx_height)**2)
        
        # 2D 柱面波解析解 (大参数 Hankel 函数渐近展开)，PEC 下边界反射系数为 -1
        E_2d = np.exp(1j * self.k0 * R1) / np.sqrt(R1) - np.exp(1j * self.k0 * R2) / np.sqrt(R2)
        E_2d_mag = np.abs(E_2d)
        
        return E_2d_mag


In [73]:
# ==========================================
# FDTD 求解器接口 (FDTD Solver)
# ==========================================
class FDTDSolver:
    def __init__(self, scene: SceneGenerator):
        self.scene = scene

    def run(self):
        mp.verbosity(0)
        
        # ── 坐标系说明 ──────────────────────────────────────────────
        # 绝对坐标 (abs): x_full 的原始坐标，范围 [0, lx+2*dpml]
        # Meep 坐标 (meep): 以仿真域中心为原点，= abs - x_center
        # 物理坐标 (phys): 相对于左侧 PML 边界，= abs - dpml
        # ────────────────────────────────────────────────────────────
        x_center = np.mean(self.scene.x_full)   # ≈ (lx + 2*dpml) / 2
        x_meep   = self.scene.x_full - x_center

        # ── 地形几何体 ──────────────────────────────────────────────
        sea_geometry = []
        floor_z = -self.scene.lz / 2 - self.scene.dpml
        for i in range(len(x_meep) - 1):
            z_val1 = min(self.scene.base_water_level + self.scene.h_full[i],
                         self.scene.lz / 2 - self.scene.dpml - 0.5)
            z_val2 = min(self.scene.base_water_level + self.scene.h_full[i + 1],
                         self.scene.lz / 2 - self.scene.dpml - 0.5)
            v1 = mp.Vector3(x_meep[i],     floor_z)
            v2 = mp.Vector3(x_meep[i + 1], floor_z)
            v3 = mp.Vector3(x_meep[i + 1], z_val2)
            v4 = mp.Vector3(x_meep[i],     z_val1)
            sea_geometry.append(mp.Prism([v1, v2, v3, v4],
                                         height=mp.inf, material=mp.metal))

        cell_size      = mp.Vector3(self.scene.lx + 2 * self.scene.dpml,
                                    self.scene.lz + 2 * self.scene.dpml)
        boundary_layers = [mp.PML(self.scene.dpml)]

        # ── 频率转换 ────────────────────────────────────────────────
        c_light      = 299792458.0
        wavelength_m = c_light / (self.scene.freq_ghz * 1e9)
        freq_meep    = 1.0 / wavelength_m

        # ── 源位置（坐标对齐修复保留）───────────────────────────────
        tx_physical_x = 10.0
        tx_abs_x      = self.scene.dpml + tx_physical_x   # 绝对坐标 = 15.0
        tx_meep_x     = tx_abs_x - x_center               # Meep 坐标
        tx_z_meep     = self.scene.base_water_level + self.scene.tx_height

        # ✅ 恢复：各向同性点源（正确的柱面波物理模型）
        sources = [mp.Source(
            mp.ContinuousSource(frequency=freq_meep),
            component=mp.Ez,
            center=mp.Vector3(tx_meep_x, tx_z_meep),
            size=mp.Vector3(0, 0)    # ✅ 点源，不是线源
        )]

        sim = mp.Simulation(
            cell_size=cell_size,
            boundary_layers=boundary_layers,
            geometry=sea_geometry,
            sources=sources,
            resolution=self.scene.resolution,
            force_complex_fields=True
        )

        steady_state_time = self.scene.lx * 5
        print(f"⏳ 开始 FDTD 仿真 (预计达到稳态时间: {steady_state_time})...")
        sim.run(until=steady_state_time)

        # ── 提取场数据 ──────────────────────────────────────────────
        ez_data = sim.get_array(center=mp.Vector3(), size=cell_size, component=mp.Ez)

        # 坐标轴：从 0 到 cell_size.x（绝对坐标）
        x_coords_full = np.linspace(0, cell_size.x, ez_data.shape[0])
        z_coords_full = np.linspace(-cell_size.y / 2, cell_size.y / 2, ez_data.shape[1])

        # ✅ 修复：提取起点 = 源的绝对坐标 tx_abs_x（而非 dpml+tx_physical_x 的旧错误）
        abs_end_x = self.scene.dpml + self.scene.lx
        tx_x_idx  = np.argmin(np.abs(x_coords_full - tx_abs_x))   # ✅ 与源位置对齐
        end_idx   = np.argmin(np.abs(x_coords_full - abs_end_x))
        z_idx     = np.argmin(np.abs(z_coords_full - tx_z_meep))   # 接收高度

        fdtd_range    = x_coords_full[tx_x_idx:end_idx] - x_coords_full[tx_x_idx]
        fdtd_1d_mag   = np.abs(ez_data[tx_x_idx:end_idx, z_idx])
        fdtd_2d_mag   = np.abs(ez_data[tx_x_idx:end_idx, :])
        z_physical_coords = z_coords_full - self.scene.base_water_level

        print(f"✅ FDTD 全波解计算完毕，有效长度: {fdtd_range[-1]:.2f}m")
        
        # ✅ 修复：返回 tx_abs_x，使 PE 侧能精确对齐起点
        return fdtd_range, fdtd_1d_mag, fdtd_2d_mag, z_physical_coords, tx_abs_x

In [74]:
# ==========================================
# PE 求解器接口 (PE Solver)
# ==========================================
class PESolver:
    def __init__(self, scene: SceneGenerator, dx=0.1, dz=0.1):
        self.c = 299792458.0
        self.freq = scene.freq_ghz * 1e9
        self.k0 = 2 * np.pi * self.freq / self.c
        self.dx = dx
        self.dz = dz
        self.max_z = scene.lz
        self.physical_lz = scene.lz
        # 增加 33% 的额外高度作为吸收层缓冲区
        self.computation_lz = scene.lz * 1.5 
        self.nz = int(self.computation_lz / dz)
        self.fft_size = 2 * self.nz 
        self.z = np.arange(self.nz) * self.dz
        self.kz = fft.fftfreq(self.fft_size, d=self.dz) * 2 * np.pi
        self.u = np.zeros(self.fft_size, dtype=np.complex128)
        self._setup_absorber()

    def _setup_absorber(self):
        self.absorber = np.ones(self.nz)
        absorb_layer_thickness = int(self.nz * 0.25)
        start_idx = self.nz - absorb_layer_thickness
        window = 0.5 * (1 + np.cos(np.pi * np.arange(absorb_layer_thickness) / absorb_layer_thickness))
        self.absorber[start_idx:] = window


        # 1. 改进 PE 初值场：引入高斯启动器减少近场震荡
    # def init_gaussian_source(self, antenna_z_phys, h_surf_0, beam_width=0.5):
    #     """
    #     使用高斯 starter 代替硬点源，beam_width 控制波束宽度
    #     """
    #     zeta_a = antenna_z_phys - h_surf_0
    #     # 构造高斯分布
    #     self.u[:self.nz] = np.exp(-((self.z - zeta_a)**2) / (2 * beam_width**2))
        
    #     # 进行频谱过滤，滤除不可传播的大角度分量
    #     kz_filter = np.exp(-(self.kz / (0.9 * self.k0))**10)
    #     self.u = fft.ifft(fft.fft(self.u) * kz_filter)


    def init_gaussian_source(self, antenna_z_phys, h_surf_0, beam_width=0.2):
        zeta_a = antenna_z_phys - h_surf_0
        # 构造高斯分布
        self.u[:self.nz] = np.exp(-((self.z - zeta_a)**2) / (2 * beam_width**2))
        
        # ✅ 修复：在进行 FFT 滤波之前，必须强制施加下边界的奇对称条件！
        self.u[self.nz + 1:] = -self.u[self.nz - 1: 0: -1]
        self.u[0] = 0.0
        self.u[self.nz] = 0.0
        
        # 进行频谱过滤
        kz_filter = np.exp(-(self.kz / (0.9 * self.k0))**10)
        self.u = fft.ifft(fft.fft(self.u) * kz_filter)

        
    def march(self, x_surf, h_surf, max_range, receiver_z_phys, smooth_window=10):
        print("⏳ 开始 PE 传播步进...")
        h_surf_smoothed = uniform_filter1d(h_surf, size=smooth_window, mode='nearest')
        

        # ✅ 修复：提前计算 physical_nz，并在初始化时就只截取物理高度
        physical_nz = int(self.max_z / self.dz)
        
        # 初始化：记录 x=0 点
        results_x    = [0.0]
        results_2d   = [np.abs(self.u[:physical_nz])] # <--- 改成 physical_nz
        h_surf_pe    = [h_surf_smoothed[0]]


        idx_rx_0 = int((receiver_z_phys - h_surf_smoothed[0]) / self.dz)
        E0 = np.abs(self.u[idx_rx_0]) if 0 <= idx_rx_0 < self.nz else 1e-12
        results_E_mag = [E0]

        steps = int(max_range / self.dx)
        for s in range(1, steps + 1):
            # ✅ 修复：正确的步进坐标
            x_curr = (s - 1) * self.dx   # 当前步起点
            x_next = s * self.dx          # 当前步终点（记录点）

            z_curr = np.interp(x_curr, x_surf, h_surf_smoothed)
            z_next = np.interp(x_next, x_surf, h_surf_smoothed)
            slope  = (z_next - z_curr) / self.dx
            beta   = np.arctan(slope)

            # --- 边界条件施加 ---
            gamma = -1.0 + 0j
            val_ref    = np.cos(beta)**2 + 0j
            refraction = np.exp(1j * self.k0 * self.dx * (np.sqrt(val_ref) - 1.0))

            self.u[:self.nz] = self.u[:self.nz] * refraction * self.absorber
            self.u[self.nz + 1:] = gamma * self.u[self.nz - 1: 0: -1]
            self.u[0]     *= (1.0 + gamma)
            self.u[self.nz] = 0.0

            # # --- 自由空间衍射传播 ---
            # k_eff_sq  = (self.k0 * np.cos(beta))**2
            # val_diff  = k_eff_sq - self.kz**2 + 0j
            # diffraction = np.exp(1j * self.dx * (np.sqrt(val_diff) - self.k0 * np.cos(beta)))

            # u_k      = fft.fft(self.u)
            # u_k      = u_k * diffraction
            # self.u   = fft.ifft(u_k)
            k_eff_sq  = (self.k0 * np.cos(beta))**2
            val_diff  = k_eff_sq - self.kz**2 + 0j
            
            # 1. 计算平方根
            sqrt_val = np.sqrt(val_diff)
            # 2. 核心修复：强制虚部为正（保证 np.exp(1j * 1j) 变成 exp(-1)，即衰减）
            sqrt_val = np.real(sqrt_val) + 1j * np.abs(np.imag(sqrt_val))
            
            diffraction = np.exp(1j * self.dx * (sqrt_val - self.k0 * np.cos(beta)))

            u_k      = fft.fft(self.u)
            
            # 3. 辅助修复：在波数域添加一个极轻微的低通滤波，进一步压制空间混叠条纹
            window_k = np.exp(-(self.kz / (0.95 * self.k0))**10)
            
            u_k      = u_k * diffraction * window_k
            self.u   = fft.ifft(u_k)

            # ✅ 修复：截断吸收层，只保留物理仿真域 [0, max_z]
            E_mag_2d = np.abs(self.u[:physical_nz])
            results_x.append(x_next)
            results_2d.append(E_mag_2d)
            h_surf_pe.append(z_next)

            zeta_rx = receiver_z_phys - z_next
            idx = int(zeta_rx / self.dz) if 0 <= zeta_rx < self.max_z else -1
            results_E_mag.append(E_mag_2d[idx] if idx != -1 else 1e-12)

        # ✅ 新增：对1D结果补偿柱面波几何扩展 1/√r
        x_arr    = np.array(results_x)
        E_arr    = np.array(results_E_mag)

        print("✅ PE 传播计算完毕")
        return (x_arr, E_arr,
                np.array(results_2d).T, self.z[:physical_nz], np.array(h_surf_pe))

In [75]:
# ==========================================
# 评估器与数学计算工具 (Metrics Evaluator)
# ==========================================
class MetricsEvaluator:
    @staticmethod
    def normalize_to_reference(range_arr, mag_arr, ref_dist=200.0, window=5.0):
        """
        将场强序列归一化：以 ref_dist 处的场强为 0 dB 基准。
        使用 window 长度进行局部平均，防止参考点陷入深衰落空点。
        """
        db_arr = 20 * np.log10(mag_arr + 1e-12)
        
        # 寻找参考点附近的索引范围
        mask = (range_arr >= ref_dist - window/2) & (range_arr <= ref_dist + window/2)
        if not np.any(mask):
            # 如果范围不足，退而求其次找最接近的点
            ref_idx = np.argmin(np.abs(range_arr - ref_dist))
            ref_val = db_arr[ref_idx]
        else:
            ref_val = np.mean(db_arr[mask])
            
        return db_arr - ref_val

    @staticmethod
    def align_and_convert_to_dB(pe_mag, ref_mag, pe_range, ref_range, method='ref_point'):
        """
        支持两种模式：
        'mean': 原有的全局均值对齐
        'ref_point': 增益归一化（推荐用于科研对比）
        """
        if method == 'ref_point':
            # 设置参考距离为 200m (远场稳定区)
            pe_norm_dB = MetricsEvaluator.normalize_to_reference(pe_range, pe_mag, ref_dist=200.0)
            ref_norm_dB = MetricsEvaluator.normalize_to_reference(ref_range, ref_mag, ref_dist=200.0)
            return pe_norm_dB, ref_norm_dB, 0.0 # Offset 此时已包含在归一化中
        else:
            # 原有的均值对齐逻辑...
            pe_dB = 20 * np.log10(pe_mag + 1e-12)
            ref_dB = 20 * np.log10(ref_mag + 1e-12)
            calib_idx = np.where(ref_range > 100.0)[0]
            pe_interp = np.interp(ref_range[calib_idx], pe_range, pe_dB)
            offset = np.mean(pe_interp) - np.mean(ref_dB[calib_idx])
            return pe_dB, ref_dB + offset, offset

    @staticmethod
    def calc_rmse_with_protection(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range, 
                                 min_range=30.0, threshold=-60.0):
        """
        计算 RMSE，严格执行：
        1. 避开 min_range 以前的近场
        2. 屏蔽低于 threshold 的极深空点（避免 log 域误差爆炸）
        """
        # 统一插值到 FDTD 的坐标系
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        
        # 构造有效计算掩模
        mask = (fdtd_range >= min_range) & (fdtd_dB_aligned > threshold)
        
        if not np.any(mask):
            return 0.0
            
        error_sq = (pe_dB_interp[mask] - fdtd_dB_aligned[mask])**2
        rmse = np.sqrt(np.mean(error_sq))
        return rmse
    
    @staticmethod
    def analyze_step_sensitivity(steps, rmse_values):
        """
        分析步长敏感度。
        计算相邻步长之间的 RMSE 变化率。
        Returns:
            diffs: 相对变化率数组，用于判断是否收敛
            slope: 在对数坐标下的斜率（收敛阶数估计）
        """
        steps = np.array(steps)
        rmse_values = np.asarray(rmse_values)
        
        # 计算相邻两次加密网格后的变化百分比
        # delta_E = |RMSE(h_i) - RMSE(h_{i+1})| / RMSE(h_i)
        relative_changes = np.abs(np.diff(rmse_values)) / rmse_values[:-1] * 100
        
        # 计算收敛阶 (Convergence Order)
        # 在双对数坐标系下，log(Error) = p * log(Step) + C
        if len(steps) > 1:
            slope, _ = np.polyfit(np.log(steps), np.log(rmse_values), 1)
        else:
            slope = 0.0
            
        return relative_changes, slope

In [76]:
# ==========================================

# 可视化组件层 (Visualizers conforming to OCP)

# ==========================================
import matplotlib.pyplot as plt
import numpy as np
from abc import ABC, abstractmethod

# 设置全局科研绘图风格
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"] + plt.rcParams["font.serif"],
    "mathtext.fontset": "stix", # 使 LaTeX 字体与正文一致
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "savefig.dpi": 300,
    "figure.autolayout": False
})

class BaseVisualizer(ABC):
    @abstractmethod
    def plot(self, *args, **kwargs):
        pass

class HeatmapVisualizer(BaseVisualizer):
    def plot(self, ref_2d_dB, test_2d_dB, ref_range, test_range, z_coords, 
             ref_title="Reference 2D Field", test_title="PE 2D Field", save_name="Fig_Heatmap.png"):
        """
        绘制 2D 空间场强热力图对比
        优化点：采用 perceptually uniform colormap (magma)，增加刻度细节，优化 colorbar 布局
        """
        fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, constrained_layout=True)
        
        vmin, vmax = -80, -20
        extent_abs = [0, 300, z_coords[0], z_coords[-1]]
        # 使用 magma 代替 jet，避免感知伪影
        cmap_style = 'magma' 

        # 绘制参考场
        im1 = axes[0].imshow(ref_2d_dB, extent=extent_abs, origin='lower', aspect='auto', 
                             cmap=cmap_style, vmin=vmin, vmax=vmax)
        axes[0].set_title(ref_title, fontweight='bold')
        axes[0].set_ylabel('Height $z$ (m)')
        axes[0].grid(True, linestyle='--', alpha=0.2)
        axes[0].set_xlim([0, 300])
        axes[0].set_ylim([z_coords[0], z_coords[-1]])

        # 绘制测试场
        im2 = axes[1].imshow(test_2d_dB, extent=extent_abs, origin='lower', aspect='auto', 
                             cmap=cmap_style, vmin=vmin, vmax=vmax)
        axes[1].set_title(test_title, fontweight='bold')
        axes[1].set_xlabel('Range $r$ (m)')
        axes[1].set_ylabel('Height $z$ (m)')
        axes[1].grid(True, linestyle='--', alpha=0.2)
        axes[1].set_xlim([0, 300])
        axes[1].set_ylim([z_coords[0], z_coords[-1]])

        # 设置刻度
        for ax in axes:
            ax.set_xticks(np.arange(0, 301, 50))
            ax.minorticks_on()

        # 统一添加 colorbar
        cbar = fig.colorbar(im1, ax=axes, orientation='vertical', pad=0.01, shrink=0.85)
        cbar.set_label('Field Strength (dB)', labelpad=10)
        cbar.set_ticks(np.arange(-80, -19, 10))
        cbar.ax.minorticks_off()

        plt.savefig(save_name, bbox_inches='tight')
        plt.close()

class LinePlotVisualizer(BaseVisualizer):
    def plot(self, range_arr, ref_1d_dB, test_1d_dB, ref_label, test_label, rmse_val, title, save_name):
        """
        绘制 1D 场强切片对比曲线
        优化点：优化线型对比，增强图例可读性，使用 LaTeX 渲染单位
        """
        fig, ax = plt.subplots(figsize=(10, 5))
        
        mask_300 = range_arr <= 300
        range_valid = range_arr[mask_300]
        ref_valid = ref_1d_dB[mask_300]
        test_valid = test_1d_dB[mask_300]

        # 绘制参考曲线（冷色调，实线）
        ax.plot(range_valid, ref_valid, color='#1f77b4', linestyle='-', linewidth=1.2, 
                label=f'{ref_label}')
        
        # 绘制测试曲线（对比色，虚线，加宽以突出差异）
        ax.plot(range_valid, test_valid, color='#d62728', linestyle='--', linewidth=1.5, 
                alpha=0.9, label=f'{test_label} (RMSE: {rmse_val:.2f} dB)')
        
        # 近场边界标记
        ax.axvline(x=30, color='gray', linestyle='-.', linewidth=0.8, alpha=0.6)
        ax.text(32, -85, 'Near-field Boundary', color='gray', fontsize=8)

        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Range $r$ (m)')
        ax.set_ylabel('Normalized Field Strength (dB)')
        
        ax.set_ylim([-50, 50])
        ax.set_xlim([0, 300])
        ax.set_xticks(np.arange(0, 301, 50))
        ax.minorticks_on()
        
        ax.grid(True, which='major', linestyle='--', alpha=0.4)
        ax.legend(loc='lower right', frameon=True, edgecolor='black', fancybox=False)
        
        plt.savefig(save_name, bbox_inches='tight')
        plt.close()

class CumulativeRMSEVisualizer(BaseVisualizer):
    def plot(self, range_arr, cum_rmse_arr, label, save_name):
        """
        绘制累积 RMSE 收敛曲线
        优化点：强化趋势线视觉特征，明确坐标轴物理意义
        """
        steps = np.asarray(steps)
        rmse_values = np.asarray(rmse_values)
        plt.figure(figsize=(8, 4.5))
        
        valid_mask = ~np.isnan(cum_rmse_arr)
        plt.plot(range_arr[valid_mask], cum_rmse_arr[valid_mask], 
                 color='black', linestyle='-', linewidth=1.5, label=label)
        
        plt.title('Cumulative RMSE Convergence', fontweight='bold')
        plt.xlabel('Range $r$ (m)')
        plt.ylabel('Cumulative RMSE (dB)')
        
        plt.xlim([range_arr[0], range_arr[-1]])
        plt.grid(True, which='major', linestyle=':', alpha=0.6)
        plt.minorticks_on()
        plt.legend(frameon=False)
        
        plt.savefig(save_name, bbox_inches='tight')
        plt.close()

class StepSensitivityVisualizer(BaseVisualizer):
    def plot(self, steps, rmse_values, step_name="$\\Delta z$", unit="m", save_name="Grid_Convergence.png"):
        """
        绘制网格收敛性双轴曲线 (优化版本)
        """
        fig, ax1 = plt.subplots(figsize=(8, 5))
        steps = np.asarray(steps)
        rmse_values = np.asarray(rmse_values)
        # 1. 颜色与样式重构：使用高对比度的双主色隔离
        color_rmse = '#08306B'    # 深海蓝 (主数据，强调严谨)
        color_change = '#CB181D'  # 莫兰迪红 (次轴数据，强调变动率)
        
        # 2. 绘制 RMSE 主曲线
        line1, = ax1.plot(steps, rmse_values, marker='s', color=color_rmse, 
                         linewidth=1.8, markersize=6, label=f'RMSE vs. {step_name}')
        
        ax1.set_xlabel(f'Step Size {step_name} ({unit})', fontweight='bold')
        ax1.set_ylabel('RMSE (dB)', color=color_rmse, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=color_rmse)
        
        ax1.set_xscale('log', base=2)
        ax1.invert_xaxis() 
        
        # 3. 绘制相对变化率次坐标轴
        if len(steps) > 1:
            ax2 = ax1.twinx()
            changes = np.abs(np.diff(rmse_values)) / rmse_values[:-1] * 100
            mid_steps = (steps[:-1] + steps[1:]) / 2
            
            # 优化柱体表现：增加不透明度，添加描边与色彩关联
            ax2.bar(mid_steps, changes, width=steps[:-1]*0.15, alpha=0.35, 
                    color=color_change, edgecolor='white', linewidth=1.2, 
                    label='Relative Change (%)')
            
            ax2.set_ylabel('Relative Change (%)', color=color_change, fontweight='bold')
            ax2.tick_params(axis='y', labelcolor=color_change)
            
            # 顶部留白，防止柱体与折线交叉干扰过大
            ax2.set_ylim([0, max(changes) * 1.35 if len(changes) > 0 else 10])

        plt.title(f'Grid Independence Test ({step_name})', fontweight='bold', pad=12)
        
        # 4. 网格线控制：降低视觉干扰
        ax1.grid(True, which="major", linestyle='-', alpha=0.15, color='black')
        ax1.grid(True, which="minor", linestyle=':', alpha=0.08, color='black')
        
        ax1.set_xticks(steps)
        ax1.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        
        plt.tight_layout()
        plt.savefig(save_name, dpi=300)
        plt.close()

In [77]:
# ==========================================
# 主流程控制 (Optimized Experiment Execution)
# ==========================================
if __name__ == "__main__":
    # 1. 实例化所有优化后的可视化组件
    heatmap_vis = HeatmapVisualizer()
    lineplot_vis = LinePlotVisualizer()
    rmse_vis = CumulativeRMSEVisualizer()
    sens_vis = StepSensitivityVisualizer()
    evaluator = MetricsEvaluator()

    # ---------------------------------------------------------
    # 实验一：退化验证（平坦海面） PE vs. Two-Ray
    # ---------------------------------------------------------
    print(f"\n{'='*60}\n▶ 实验一：退化验证 (PE vs. Two-Ray 解析解)\n{'='*60}")
    
    scene_flat = SceneGenerator(terrain_type='flat', lx=300.0)
    
    # PE 求解
    tx_physical_x = 10.0
    tx_idx = np.argmin(np.abs(scene_flat.x_full - tx_physical_x))
    pe_x_input = scene_flat.x_full[tx_idx:] - scene_flat.x_full[tx_idx]
    pe_h_input = scene_flat.h_full[tx_idx:]
    
    pe_solver_flat = PESolver(scene_flat, dz=0.1, dx=0.2)
    pe_solver_flat.init_gaussian_source(scene_flat.tx_height, pe_h_input[0])
    pe_range_flat, pe_1d_flat, pe_2d_raw_flat, z_coords_pe_flat, h_surf_flat = pe_solver_flat.march(
        pe_x_input, pe_h_input, 290.0, scene_flat.rx_height
    )
    
    # Two-Ray 求解
    tr_solver = TwoRaySolver(scene_flat)
    z_coords_abs = np.linspace(0, scene_flat.lz, pe_2d_raw_flat.shape[0])
    tr_2d_mag = tr_solver.run(pe_range_flat, z_coords_abs)
    rx_z_idx = np.argmin(np.abs(z_coords_abs - scene_flat.rx_height))
    tr_1d_flat = tr_2d_mag[:, rx_z_idx]
    
    # 归一化与误差计算 (使用 ref_point 消除源增益偏差)
    pe_1d_dB_flat, tr_1d_dB_flat, _ = evaluator.align_and_convert_to_dB(
        pe_1d_flat, tr_1d_flat, pe_range_flat, pe_range_flat, method='ref_point'
    )
    rmse_flat = evaluator.calc_rmse_with_protection(
        pe_1d_dB_flat, tr_1d_dB_flat, pe_range_flat, pe_range_flat, min_range=30.0
    )
    print(f"✅ 平坦海面归一化 RMSE: {rmse_flat:.4f} dB")
    
    # 2D 场绘图准备 (采用相同的归一化 Offset)
    heatmap_vis.plot(20*np.log10(tr_2d_mag.T + 1e-12), 
                     20*np.log10(pe_2d_raw_flat + 1e-12), 
                     pe_range_flat, pe_range_flat, z_coords_abs, 
                     ref_title="Two-Ray Analytical Field", test_title="PE Model (Degraded)", 
                     save_name="Exp1_Heatmap_Flat.png")
    
    lineplot_vis.plot(pe_range_flat, tr_1d_dB_flat, pe_1d_dB_flat, 
                      "Two-Ray", "PE (Flat)", rmse_flat, 
                      "1D Comparison: Flat Surface", "Exp1_LinePlot_Flat.png")


    # =========================================================
    # 实验二：网格无关性验证 (PE 自收敛测试，2m/s 中等海况)
    # =========================================================
    print(f"\n{'='*60}\n▶ 实验二：网格收敛性分析 (Grid Convergence Study)\n{'='*60}")
    
    # 1. 生成统一的粗糙海面 (保证所有网格下地形绝对一致)
    scene_rough = SceneGenerator(lx=300.0, wind_speed=2.0)
    
    # 获取海面切片输入
    tx_physical_x = 10.0
    tx_idx = np.argmin(np.abs(scene_rough.x_full - tx_physical_x))
    pe_x_input = scene_rough.x_full[tx_idx:] - scene_rough.x_full[tx_idx]
    pe_h_input = scene_rough.h_full[tx_idx:]

    # 2. 计算“拟似真实解 (Ground Truth)” -> 使用极细网格 dz = 0.05
    print("⏳ 正在计算拟似真实解 (Ground Truth), Δz = 0.05m ...")
    pe_solver_gt = PESolver(scene_rough, dz=0.05, dx=0.2)
    pe_solver_gt.init_gaussian_source(scene_rough.tx_height, pe_h_input[0], beam_width=0.2)
    r_gt, e_1d_gt, _, _, _ = pe_solver_gt.march(pe_x_input, pe_h_input, 290.0, scene_rough.rx_height)
    
    # 距离补偿与幅度转对数
    e_1d_gt_comp = e_1d_gt / np.sqrt(np.maximum(r_gt, 1.0))
    e_1d_gt_dB = 20 * np.log10(e_1d_gt_comp + 1e-12)

    # 3. 遍历其他网格进行对比测试
    dz_steps = [0.8, 0.4, 0.2, 0.1, 0.05]
    rmse_vs_dz = []
    
    for dz_test in dz_steps:
        if dz_test == 0.05:
            rmse_vs_dz.append(0.0)  # 自己和自己比误差为 0
            continue
            
        print(f"⏳ 正在测试网格步长 Δz = {dz_test}m ...")
        test_solver = PESolver(scene_rough, dz=dz_test, dx=0.2)
        test_solver.init_gaussian_source(scene_rough.tx_height, pe_h_input[0], beam_width=0.2)
        r_test, e_1d_test, _, _, _ = test_solver.march(pe_x_input, pe_h_input, 290.0, scene_rough.rx_height)
        
        e_1d_test_comp = e_1d_test / np.sqrt(np.maximum(r_test, 1.0))
        
        # 归一化对齐：消除不同 dz 下初始源可能带来的增益偏差，只比对干涉波纹分布误差
        test_dB, gt_aligned_dB, _ = evaluator.align_and_convert_to_dB(
            e_1d_test_comp, e_1d_gt_comp, r_test, r_gt, method='mean'
        )
        
        # 计算 RMSE (屏蔽掉前 30m 近场区的极大值)
        rmse = evaluator.calc_rmse_with_protection(test_dB, gt_aligned_dB, r_test, r_gt, min_range=30.0)
        rmse_vs_dz.append(rmse)
        print(f"   ✅ RMSE = {rmse:.4f} dB")

    # 4. 可视化输出及分析
    sens_vis = StepSensitivityVisualizer()
    sens_vis.plot(dz_steps, rmse_vs_dz, step_name="$\\Delta z$", unit="m", save_name="Exp2_Grid_Convergence.png")
    
    changes, slope = evaluator.analyze_step_sensitivity(dz_steps, rmse_vs_dz)
    print(f"\n✅ 网格无关性测试完成！")
    print(f"   - 对数收敛斜率 (Log-Log Slope): {slope:.2f}")
    if len(changes) > 0:
        # 查看从 0.2m 加密到 0.1m 时的相对变化百分比
        print(f"   - 在 Δz = 0.2m 时的相对误差收敛度: {changes[-2]:.2f}%")
        print("   -> 结论：兼顾效率与精度，推荐在后续复杂海况全波仿真中采用 Δz = 0.2m。")

    print(f"\n{'='*60}\n🎉 所有实验已完成。优化后的科研图表已保存至当前目录。\n{'='*60}")


▶ 实验一：退化验证 (PE vs. Two-Ray 解析解)
⏳ 开始 PE 传播步进...
✅ PE 传播计算完毕
✅ 平坦海面归一化 RMSE: 0.5611 dB

▶ 实验二：网格收敛性分析 (Grid Convergence Study)
⏳ 正在计算拟似真实解 (Ground Truth), Δz = 0.05m ...
⏳ 开始 PE 传播步进...
✅ PE 传播计算完毕
⏳ 正在测试网格步长 Δz = 0.8m ...
⏳ 开始 PE 传播步进...
✅ PE 传播计算完毕
   ✅ RMSE = 2.0749 dB
⏳ 正在测试网格步长 Δz = 0.4m ...
⏳ 开始 PE 传播步进...
✅ PE 传播计算完毕
   ✅ RMSE = 1.0575 dB
⏳ 正在测试网格步长 Δz = 0.2m ...
⏳ 开始 PE 传播步进...
✅ PE 传播计算完毕
   ✅ RMSE = 1.0759 dB
⏳ 正在测试网格步长 Δz = 0.1m ...
⏳ 开始 PE 传播步进...
✅ PE 传播计算完毕
   ✅ RMSE = 0.4044 dB

✅ 网格无关性测试完成！
   - 对数收敛斜率 (Log-Log Slope): nan
   - 在 Δz = 0.2m 时的相对误差收敛度: 62.42%
   -> 结论：兼顾效率与精度，推荐在后续复杂海况全波仿真中采用 Δz = 0.2m。

🎉 所有实验已完成。优化后的科研图表已保存至当前目录。


/tmp/ipykernel_4707/4038365159.py:85: RuntimeWarning: divide by zero encountered in log
  slope, _ = np.polyfit(np.log(steps), np.log(rmse_values), 1)
